In [ ]:
from collections import defaultdict
import cv2
import numpy as np
from ultralytics import YOLO
import csv
import os
from collections import defaultdict, Counter

EXIT_MARGIN_RATIO = 0.3  # 30% of frame width
MIN_TRACK_LENGTH = 3    # minimum track length to consider for counting

def get_zone(cx, frame_width, margin_ratio=EXIT_MARGIN_RATIO):
    if cx <= frame_width * margin_ratio:               # left boundary
        return "left"
    elif cx >= frame_width * (1 - margin_ratio):       # right boundary
        return "right"
    else:
        return "middle"
    
def decide_track_class(class_history):
    """Majority vote across the track's per-frame class votes."""
    return Counter(class_history).most_common(1)[0][0]

def classify_track(track, width, dominant_flow):
    positions = track["positions"]

    if len(positions) < 3:
        return "pending", "unknown", "too_short"

    xs = [p[1] for p in positions]

    first_x = xs[0]
    last_x = xs[-1]
    dx = last_x - first_x

    first_side = track["first_side"]
    exit_side = track["last_side"]

    MIN_DISPLACEMENT = 0.25 * width

    # If entry side is reliable, use zones
    if first_side in ["left", "right"]:
        if first_side != exit_side:
            return "confirmed", f"{first_side}_to_{exit_side}", "opposite_exit"
        else:
            return "pending", "unknown", "same_side_not_traversal"

    # If entry side is middle, use displacement
    if abs(dx) < MIN_DISPLACEMENT:
        return "pending", "unknown", "not_enough_displacement"

    if dx > 0 and exit_side == "right":
        if dominant_flow in ["Unknown", "Left -> Right"]:
            return "confirmed", "left_to_right", "middle_entry_matches_flow"
        else:
            return "pending", "unknown", "middle_entry_conflicts_with_flow"
    if dx < 0 and exit_side == "left":
        if dominant_flow in ["Unknown", "Right -> Left"]:
            return "confirmed", "right_to_left", "middle_entry_matches_flow"
        else:
            return "pending", "unknown", "middle_entry_conflicts_with_flow"

    return "pending", "unknown", "displacement_exit_conflict"

# visualization of tracks and zones --- IGNORE ---
def visualization(annotated, classwise_track_ids, dominant_flow, tracks, fw, fh):
    font= cv2.FONT_HERSHEY_SIMPLEX
    scale     = fh / 720
    thickness = max(1, int(scale))
    margin    = 10
    text_h    = cv2.getTextSize("Non-Herring: 9999", font, scale, thickness)[0][1]
    text_w    = cv2.getTextSize("Non-Herring: 9999", font, scale, thickness)[0][0]
    line_gap = text_h + 8

    x0        = fw - margin - text_w
    y0        = margin + text_h

    cv2.putText(
        annotated,
        f"Herring: {len(classwise_track_ids['Herring'])}",
        (x0, y0), font, scale, (0,255,0), thickness
    )
    cv2.putText(
        annotated,
        f"Non-Herring: {len(classwise_track_ids['Non-Herring'])}",
        (x0, y0 + text_h + 5), font, scale, (0,255,255), thickness
    )
    cv2.putText(
        annotated,
        f"Flow: {dominant_flow}",
        (x0, y0 + (text_h + 5) * 2), font, scale, (255, 255, 0), thickness
    )

    # top left corner: counted & pending track IDs
    counted_ids = sorted([
        tid for tid, t in tracks.items()
        if t["counted"]
    ])

    pending_ids = sorted([
        tid for tid, t in tracks.items()
        if t["status"] == "pending"
    ])

    def split_ids(ids, chunk_size=10):
        return [
            ids[i:i + chunk_size]
            for i in range(0, len(ids), chunk_size)
        ]
    
    counted_text = split_ids(counted_ids, 10)
    pending_text = split_ids(pending_ids, 10)

    y = margin + line_gap

    # Counted IDs
    for i, chunk in enumerate(counted_text):
        text = "Counted IDs: " if i == 0 else "            "
        text += ", ".join(map(str, chunk))

        cv2.putText(
            annotated,
            text,
            (margin, y),
            font,
            scale,
            (0, 255, 0),
            thickness
        )
        y += line_gap

    # Small gap between sections
    y += 5

    # Pending IDs
    for i, chunk in enumerate(pending_text):
        text = "Pending IDs: " if i == 0 else "            "
        text += ", ".join(map(str, chunk))

        cv2.putText(
            annotated,
            text,
            (margin, y),
            font,
            scale,
            (0, 165, 255),
            thickness
        )
        y += line_gap

    

def fish_counting_driver(model, input_video_path, output_video_path, output_csv_path):
    # --- initialize video capture and writer ---
    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open video: {input_video_path}")
    
    # Get video properties
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Define the codec and create VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    # Open CSV file for writing
    csv_file = open(output_csv_path, mode='w', newline='')
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow([
        "frame_id",
        "track_id",
        "class_name",
        "confidence",
        "center_x", "center_y",
        "first_side",
        "last_side",
        "direction",
        "status",
        "reason",
    ])
        
    frame_id = -1

    # 2) Buffers & bookkeeping
    tracks = {}
    classwise_track_ids  = {'Herring': set(), 'Non-Herring': set()}   # final counts
    direction_counts = Counter({
    "left_to_right": 0,
    "right_to_left": 0
    })

    dominant_flow = "Unknown"

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break
        frame_id += 1

        # --- run detection + tracking ---
        results = model.track(frame, persist=True, tracker="botsort.yaml", verbose=False)

        # --- annotated frame for visualization ---
        if results[0].boxes is not None and results[0].boxes.id is not None:
            annotated_frame = frame.copy() # custom annotation
            xywh = results[0].boxes.xywh.cpu().tolist()
            ids = results[0].boxes.id.int().cpu().tolist()
            confidences = results[0].boxes.conf.cpu().tolist()
            class_ids = results[0].boxes.cls.int().cpu().tolist()

            # --- update track states ---
            for (x_c, y_c, w, h), tid, conf, cid in zip(xywh, ids, confidences, class_ids):
                cname = model.names[cid]
                current_side = get_zone(x_c, width)

                if tid not in tracks:
                    tracks[tid] = {
                        "positions": [(frame_id, x_c, y_c)], #[(frame_id, x_c, y_c)],
                        "first_side": current_side, # left / right / middle
                        "last_side": current_side,  # left / right / middle
                        "direction": "unknown",  # left_to_right / right_to_left / unknown
                        "status": "tracking",     # tracking / pending / confirmed / rejected
                        "class_history": [cname],
                        "counted": False,          # whether this track has been counted in the final tally
                    }

                else:
                    tracks[tid]["positions"].append((frame_id, x_c, y_c))
                    tracks[tid]["class_history"].append(cname)
                    tracks[tid]["last_side"] = current_side

                # ─── Real-time counting check ────────
                if not tracks[tid]["counted"] and current_side in ["left", "right"]:
                    status, direction, reason = classify_track(tracks[tid], width, dominant_flow)

                    if status == "confirmed":
                        final_class = decide_track_class(tracks[tid]["class_history"])

                        if final_class in classwise_track_ids:
                            classwise_track_ids[final_class].add(tid)
                        else:
                            classwise_track_ids[final_class] = {tid}

                        direction_counts[direction] += 1
                        
                        tracks[tid]["counted"] = True
                        tracks[tid]["status"] = "confirmed"
                        tracks[tid]["direction"] = direction
                        tracks[tid]["reason"] = reason

                csv_writer.writerow([
                    frame_id,
                    tid,
                    cname,
                    conf,
                    x_c, y_c,
                    first_side := tracks[tid]["first_side"],
                    last_side := tracks[tid]["last_side"],
                    direction := tracks[tid]["direction"],
                    status := tracks[tid]["status"],
                    reason := tracks[tid].get("reason", "N/A"),
                ])

                # choose bbox color based on track status
                if tracks[tid]["counted"]:
                    box_color = (0, 0, 255)      # red = counted
                elif tracks[tid]["status"] == "pending":
                    box_color = (128, 128, 128)  # grey = pending
                elif tracks[tid]["first_side"] in ["left", "right"]:
                    box_color = (255, 0, 0)      # blue = entered from left/right
                else:
                    box_color = (128, 128, 128)  # grey = middle/unknown entry

                # convert xywh to xyxy
                x1 = int(x_c - w / 2)
                y1 = int(y_c - h / 2)
                x2 = int(x_c + w / 2)
                y2 = int(y_c + h / 2)

                cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), box_color, 2)

                label = f"ID {tid} {tracks[tid]['status']} {cname} {conf:.2f}"
                cv2.putText(
                    annotated_frame,
                    label,
                    (x1, max(20, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    box_color,
                    2
                )
        
        else:
            annotated_frame = frame.copy()
            xywh, ids, confidences, class_ids = [], [], [], []

        fh, fw = annotated_frame.shape[:2]
        EXIT_MARGIN_PX = int(fw * EXIT_MARGIN_RATIO)

        # draw exit zones
        cv2.line(annotated_frame, (EXIT_MARGIN_PX, 0), (EXIT_MARGIN_PX, fh), (0, 0, 255), 2)
        cv2.line(annotated_frame, (fw - EXIT_MARGIN_PX, 0), (fw - EXIT_MARGIN_PX, fh), (0, 0, 255), 2)

        ltr = direction_counts["left_to_right"]
        rtl = direction_counts["right_to_left"]

        if ltr > rtl:
            dominant_flow = "Left -> Right"
        elif rtl > ltr:
            dominant_flow = "Right -> Left"
        else:
            dominant_flow = "Unknown"

        # Draw counts top right
        visualization(annotated_frame, classwise_track_ids, dominant_flow, tracks, fw, fh)

        out.write(annotated_frame)

    cap.release()
    out.release()
    csv_file.close()

    print("Done.")

    
# --- Parameters (fill in directly, or override via environment variables) ---
# Env-var overrides let this same notebook be batched over multiple
# (video, model) combinations without editing the cell each time.
yolo_model_path   = os.environ.get("YOLO_MODEL_PATH", "")
input_video_path  = os.environ.get("INPUT_VIDEO_PATH", "")
output_path_prefix = os.environ.get("OUTPUT_PATH_PREFIX", "")

output_video_path = f"{output_path_prefix}.mp4"
output_csv_path   = f"{output_path_prefix}.csv"

yolo_model = YOLO(yolo_model_path)  # load a custom model

# NOTE: use mp4 video format, or change the codec accordingly!
fish_counting_driver(
    yolo_model,
    input_video_path,
    output_video_path,
    output_csv_path
    )

